# テイラー展開 2：行列の指数関数

$e^x=\sum\frac{x^k}{k!}$ の $x$ に**正方行列** $A$ を入れたものが、行列の指数関数

$$
e^A=\sum_{k=0}^\infty\frac{A^k}{k!}=I+A+\frac{A^2}{2!}+\frac{A^3}{3!}+\cdots
$$

です（行列の掛け算と足し算だけで書けるので、そのまま意味を持ちます）。[関数・線形写像・微分・積分のノート](../../../研究ノート/01_基礎・ベクトル解析/functions_linear_maps_derivatives_integrals.md)で扱い、パウリ行列・$SU(2)$ のノートで何度も使ってきました。このノートブックでは、

1. 級数の部分和が収束する様子
2. 行列版のオイラーの公式：$e^{\theta J}$ は回転行列
3. 挟み込み $Pe^AP^{-1}=e^{PAP^{-1}}$ と、対角化による計算
4. 微分は1次の近似：行列式の場合（$O(\varepsilon^2)$ の意味）
5. $\det(e^A)=e^{\operatorname{tr}A}$ と、Lean による証明
6. $e^{A+B}\ne e^Ae^B$（交換しないとき）
7. トロッター分解：量子コンピュータで時間発展を作る方法

を見ます。

In [ ]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as W
from math import factorial
from numpy.typing import NDArray
from plotly.subplots import make_subplots

Floats = NDArray[np.float64]       # 実数の配列（型ヒント用の別名）
Complexes = NDArray[np.complex128]
# 注意：FigureWidget（スライダーで描き直す図）には、numpy 配列ではなくリストを渡す（NB-02 を参照）
from scipy.linalg import expm

Matrix = NDArray[np.complex128]    # 正方行列（型ヒント用の別名）


def exp_series(A: Matrix, N: int) -> Matrix:
    """N 次までの部分和 Σ_{k=0}^{N} A^k / k!"""
    total = np.zeros_like(A, dtype=complex)
    term = np.eye(A.shape[0], dtype=complex)
    for k in range(N + 1):
        total = total + term
        term = term @ A / (k + 1)
    return total

## 1. 部分和は収束する

ランダムな $3\times3$ 行列で、部分和と scipy の `expm`（行列の指数関数を精度よく計算する関数）の差を、次数 $N$ に対して描きます。行列の「大きさ」（ノルム）が大きいほど、収束に多くの項が必要です。

In [ ]:
rng = np.random.default_rng(0)
A_small = rng.normal(size=(3, 3)) * 0.5
A_large = rng.normal(size=(3, 3)) * 3.0
Ns = np.arange(0, 61)
fig = go.Figure()
for A, label in ((A_small, "小さい行列（ノルム ≈ %.1f）" % np.linalg.norm(A_small, 2)),
                 (A_large, "大きい行列（ノルム ≈ %.1f）" % np.linalg.norm(A_large, 2))):
    err = [float(np.linalg.norm(exp_series(A, int(N)) - expm(A))) for N in Ns]
    fig.add_trace(go.Scatter(x=Ns.tolist(), y=err, mode="lines", name=label))
    assert np.allclose(exp_series(A, 60), expm(A))
fig.update_layout(height=360, width=780, yaxis_type="log", xaxis_title="次数 N", yaxis_title="expm との差", margin=dict(t=20))
fig

## 2. 行列版のオイラーの公式：$e^{\theta J}$ は回転行列

$J=\begin{pmatrix}0&-1\\1&0\end{pmatrix}$ は $J^2=-I$ を満たし、複素数の $i$ と同じ役割をします（[複素数の幾何学のノート](../../../研究ノート/04_群論・代数/complex_numbers_geometry.md)の Part III・IV）。$e^x$ の級数で $x=\theta J$ とすると、$J^2=-I$ から偶数次と奇数次に分かれて

$$
e^{\theta J}=\cos\theta\,I+\sin\theta\,J=\begin{pmatrix}\cos\theta&-\sin\theta\\ \sin\theta&\cos\theta\end{pmatrix}
$$

と、**回転行列**になります。1冊目のオイラーの公式 $e^{i\theta}=\cos\theta+i\sin\theta$ の、行列版です。スライダーで $\theta$ を変え、級数で計算した $e^{\theta J}$ で正方形を回します。

In [ ]:
J = np.array([[0, -1], [1, 0]], dtype=complex)
assert np.allclose(J @ J, -np.eye(2))
for theta in np.linspace(0, 2 * np.pi, 9):
    rot = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    assert np.allclose(exp_series(theta * J, 40), rot)

square = np.array([[0, 0], [1, 0], [1, 1], [0, 1], [0, 0]], dtype=float).T
fig_rot = go.FigureWidget([go.Scatter(x=square[0].tolist(), y=square[1].tolist(), fill="toself", mode="lines",
                                      line=dict(color="#aaa"), name="元の正方形"),
                           go.Scatter(x=[], y=[], fill="toself", mode="lines", line=dict(color="#1f77b4"), name="e^(θJ) で回したもの")])
fig_rot.update_layout(height=460, width=520, margin=dict(t=20), xaxis=dict(range=[-1.6, 1.6]),
                      yaxis=dict(range=[-1.6, 1.6], scaleanchor="x"))
theta_r = W.FloatSlider(value=40, min=0, max=360, step=5, description="θ [度]")
N_r = W.IntSlider(value=3, min=0, max=20, description="次数 N")
info_r = W.HTML()


def update_rot(*_: object) -> None:
    """θ と次数が変わるたびに、部分和で回した正方形を描き直す"""
    theta, N = np.radians(theta_r.value), N_r.value
    M = exp_series(theta * J, N).real
    P = M @ square
    with fig_rot.batch_update():
        fig_rot.data[1].x, fig_rot.data[1].y = P[0].tolist(), P[1].tolist()
    info_r.value = (f"N = {N} 次までの部分和：det = {np.linalg.det(M):.4f}（回転なら 1）、"
                    f"回転行列との差 = {np.linalg.norm(M - expm(theta * J).real):.2e}")


for w in (theta_r, N_r):
    w.observe(update_rot, "value")
update_rot()
W.VBox([W.HBox([theta_r, N_r]), info_r, fig_rot])

**考えてみよう**：次数 $N$ が小さいと、正方形は回るだけでなく、伸びたり歪んだりします（行列式が $1$ からずれる）。$N$ を上げると、ちょうど回転になります。パウリ行列の $e^{-i\theta\sigma/2}=\cos\frac\theta2I-i\sin\frac\theta2\sigma$（[NB-02](../../02_su2_rotation/02_bloch_rotation.ipynb)）も、$\sigma^2=I$ を使った同じ仕組みです。

## 3. 挟み込み：$Pe^AP^{-1}=e^{PAP^{-1}}$

$P$ を正則な行列とすると、$P^{-1}P=I$ が間で消えるので

$$
(PAP^{-1})^k=PAP^{-1}\,PAP^{-1}\cdots PAP^{-1}=PA^kP^{-1}
$$

です。級数の各項でこれを使うと

$$
e^{PAP^{-1}}=\sum_k\frac{PA^kP^{-1}}{k!}=P\Big(\sum_k\frac{A^k}{k!}\Big)P^{-1}=Pe^AP^{-1}
$$

です。特に $A=PDP^{-1}$（$D$ は固有値を並べた対角行列）と対角化できれば、$e^A=Pe^DP^{-1}=P\operatorname{diag}(e^{\lambda_1},\dots,e^{\lambda_n})P^{-1}$ と、固有値の指数関数だけで計算できます（[det のノート](../../../研究ノート/04_群論・代数/det_exp_trace_proof.md)の証明1）。対角化できない場合（ジョルダン標準形）の考え方と計算は、[行列の指数関数と交換子のノート](../../../研究ノート/04_群論・代数/matrix_exponential_commutator_bch_trotter.md)の Part II にまとめました。

In [ ]:
A = rng.normal(size=(3, 3)) + 1j * rng.normal(size=(3, 3))
P = rng.normal(size=(3, 3)) + 1j * rng.normal(size=(3, 3))
Pinv = np.linalg.inv(P)
for k in range(6):
    assert np.allclose(np.linalg.matrix_power(P @ A @ Pinv, k), P @ np.linalg.matrix_power(A, k) @ Pinv)
assert np.allclose(P @ expm(A) @ Pinv, expm(P @ A @ Pinv))

lam, V = np.linalg.eig(A)                            # A = V diag(λ) V⁻¹
via_diag = V @ np.diag(np.exp(lam)) @ np.linalg.inv(V)
assert np.allclose(via_diag, expm(A))
print("固有値 λ =", np.round(lam, 3))
print("V diag(e^λ) V⁻¹ と expm(A) の差 =", f"{np.linalg.norm(via_diag - expm(A)):.1e}")

# 対角化できない行列（ジョルダン細胞）：A = λI + N、N² = 0 なので e^A = e^λ (I + N)
lam0 = 0.7
Jd = np.array([[lam0, 1], [0, lam0]], dtype=complex)
assert np.allclose(expm(Jd), np.exp(lam0) * np.array([[1, 1], [0, 1]]))
print("ジョルダン細胞 [[0.7, 1], [0, 0.7]] の e^A =\n", np.round(expm(Jd).real, 4), "= e^0.7 × [[1, 1], [0, 1]]")

## 4. 微分は1次の近似：行列式の場合

微分は「1次の線形近似の係数」です：

$$
f(x+\varepsilon)=f(x)+f'(x)\,\varepsilon+O(\varepsilon^2)
$$

$O(\varepsilon^2)$（ランダウの記号、「オーダー・イプシロン2乗」）は、$\varepsilon^2$ と同じかそれより速く $0$ に近づく量です。行列式で確かめます。$\det(M+\varepsilon X)$ を $\varepsilon$ の多項式として展開すると、

- **0次**：$\det M$
- **1次**：$\varepsilon\cdot d(\det M)$。[det のノート](../../../研究ノート/04_群論・代数/det_exp_trace_proof.md)の Jacobi の公式で、$d(\det M)=\det M\cdot\operatorname{tr}(M^{-1}X)$
- **2次以上**：$X$ の成分どうしの積を含む「余分な項」。微分（1次の近似）では捨てる部分

です。$2\times2$ では、2次の項がちょうど $\det X$ になり、$\det(M+X)-\det M-\det X=d(\det M)$ が**近似でなく、ぴったり**成り立ちます。$3\times3$ 以上では、2次から $(n-1)$次の「混ざった項」も出て、$\det X$ は最も高い $n$次の項になります。

In [ ]:
import sympy as sp

eps = sp.symbols("varepsilon")
for n in (2, 3):
    M = sp.Matrix(n, n, lambda i, j: sp.Symbol(f"m{i + 1}{j + 1}"))
    X = sp.Matrix(n, n, lambda i, j: sp.Symbol(f"x{i + 1}{j + 1}"))
    poly = sp.Poly(sp.expand((M + eps * X).det()), eps)
    c = [sp.expand(poly.coeff_monomial(eps**k)) for k in range(n + 1)]
    assert sp.expand(c[0] - M.det()) == 0                                   # 0次 = det M
    assert sp.expand(c[1] - (M.adjugate() * X).trace()) == 0                # 1次 = tr(adj(M) X) = det M tr(M⁻¹X)
    assert sp.expand(c[n] - X.det()) == 0                                   # 最高次 = det X
    print(f"{n}×{n}：")
    for k in range(n + 1):
        name = {0: "det M", 1: "d(det M) = tr(adj(M) X)", n: "det X"}.get(k, "混ざった項")
        print(f"  ε^{k} の係数（{name}）：項の数 {len(sp.Add.make_args(c[k]))}")
    if n == 2:
        print("  → 2×2 では 2次の項 = det X なので、det(M+X) − det M − det X = d(det M) がぴったり成り立つ")
    else:
        print("  2次の項の例：", sp.Add.make_args(c[2])[:3], "…（M と X の成分が混ざる）")

In [ ]:
# 1次の近似の誤差が ε² で小さくなること（両対数で傾き 2）
Mn = rng.normal(size=(3, 3))
Xn = rng.normal(size=(3, 3))
d_det = np.linalg.det(Mn) * np.trace(np.linalg.inv(Mn) @ Xn)               # Jacobi の公式
eps_vals = np.logspace(-6, -1, 20)
err = np.array([abs(np.linalg.det(Mn + e * Xn) - np.linalg.det(Mn) - e * d_det) for e in eps_vals])
slope = np.polyfit(np.log(eps_vals), np.log(err), 1)[0]
print(f"両対数の傾き = {slope:.3f}（O(ε²) なら 2）")
assert abs(slope - 2) < 0.1
fig = go.Figure([go.Scatter(x=eps_vals.tolist(), y=err.tolist(), mode="lines+markers", name="|det(M+εX) − det M − ε d(det M)|"),
                 go.Scatter(x=eps_vals.tolist(), y=(err[-1] * (eps_vals / eps_vals[-1]) ** 2).tolist(), mode="lines",
                            line=dict(dash="dash"), name="ε² に比例する線")])
fig.update_layout(height=360, width=780, xaxis_type="log", yaxis_type="log", xaxis_title="ε", yaxis_title="1次の近似の誤差",
                  margin=dict(t=20))
fig

## 5. $\det(e^A)=e^{\operatorname{tr}A}$

[det のノート](../../../研究ノート/04_群論・代数/det_exp_trace_proof.md)で2通りに証明した公式を、対角化できない行列や複素行列も含めて数値で確かめます。$\operatorname{tr}A=0$ なら $\det e^A=1$ で、これが $SU(n)$ の生成子がトレースゼロになる理由でした。

この公式は、Lean 4 + Mathlib でも証明しました（Mathlib の v4.28.0 ではまだ TODO になっている定理です）：

- [DetExpTrace.lean](../../../lean4/QuantumStudy/DetExpTrace.lean) の `det_exp`：ノートの証明2の道筋（$f(t)=\det e^{tA}$ が $f'=(\operatorname{tr}A)f$ を満たす。上の §4 の「行列式の1次の近似」を、単位行列の場所で使う）
- [Schur.lean](../../../lean4/QuantumStudy/Schur.lean) の `det_exp_via_schur`：シューア分解 $A=UTU^*$（$T$ は上三角）を使う道筋

```lean
theorem det_exp (A : Matrix n n ℂ) : (exp A).det = Complex.exp A.trace
```

In [ ]:
for trial in range(200):
    A = rng.normal(size=(4, 4)) + 1j * rng.normal(size=(4, 4))
    assert np.isclose(np.linalg.det(expm(A)), np.exp(np.trace(A)))
assert np.isclose(np.linalg.det(expm(Jd)), np.exp(np.trace(Jd)))            # 対角化できない行列でも
H = rng.normal(size=(3, 3)) + 1j * rng.normal(size=(3, 3))
H = H + H.conj().T
H = H - np.trace(H) / 3 * np.eye(3)                                          # トレースゼロのエルミート行列
assert np.isclose(np.linalg.det(expm(1j * H)), 1)
print("ランダムな 4×4 複素行列 200 個、ジョルダン細胞、トレースゼロのエルミート行列で確認しました")

## 6. $e^{A+B}\neq e^Ae^B$（交換しないとき）

数なら $e^{a+b}=e^ae^b$ ですが、行列では次のようになります。

- $AB=BA$（$[A,B]=0$）なら、$e^{A+B}=e^Ae^B$ が必ず成り立ちます（**十分条件**。数のときと同じく、二項展開がそのまま使えるから）。
- **すべての $t$ で** $e^{t(A+B)}=e^{tA}e^{tB}$ が成り立つのは、$[A,B]=0$ のときだけです。両辺を $t$ で展開して $t^2$ の係数を比べると、$\frac12(A+B)^2$ と $\frac12A^2+AB+\frac12B^2$ の差が $\frac12[A,B]$ だからです。
- ただし、**特定の1点（たとえば $t=1$）だけなら、交換しなくても偶然成り立つ**ことがあります（下の反例）。

**交換子 $[A,B]=AB-BA$ と量子力学**：交換子は、量子力学の中心にある量です。物理量（行列・演算子）が交換しないことが、不確定性関係や、測定の順序で結果が変わることの源です。パウリ行列（スピン、1量子ビットの演算子）では

$$
[\sigma_x,\sigma_y]=2i\sigma_z,\qquad[\sigma_y,\sigma_z]=2i\sigma_x,\qquad[\sigma_z,\sigma_x]=2i\sigma_y\qquad\big(\text{まとめて}\ [\sigma_i,\sigma_j]=2i\varepsilon_{ijk}\sigma_k\big)
$$

で、反交換子 $\{A,B\}=AB+BA$ は $\{\sigma_i,\sigma_j\}=2\delta_{ij}I$ です（クリフォード代数の関係）。$AB=\frac12\{A,B\}+\frac12[A,B]$ なので、2つを合わせると、積がまるごと

$$
\sigma_i\sigma_j=\delta_{ij}I+i\varepsilon_{ijk}\sigma_k
$$

と書けます（$\varepsilon_{ijk}$ はレヴィ・チヴィタ記号）。導出は[パウリ行列のノート](../../../研究ノート/04_群論・代数/pauli_matrices_derivation_and_group.md)の Part IV・V、Lean による証明は [Pauli.lean](../../../lean4/QuantumStudy/Pauli.lean) にあります。下の計算で $A=\sigma_x$、$B=\sigma_z$ を使うのは、$[\sigma_x,\sigma_z]=-2i\sigma_y\ne0$ と、交換しない一番簡単な例だからです。量子コンピュータでは、交換しない項の和のハミルトニアン（$\sigma_x+\sigma_z$ など）の時間発展を作るときに、このずれが問題になります（§7）。

小さな $t$ では

$$
e^{tA}e^{tB}=e^{t(A+B)+\frac{t^2}2[A,B]+O(t^3)}
$$

（ベイカー・キャンベル・ハウスドルフの公式の最初の項。導出は[行列の指数関数と交換子のノート](../../../研究ノート/04_群論・代数/matrix_exponential_commutator_bch_trotter.md)の Part IV と付録B）なので、ずれは $\frac{t^2}2\|[A,B]\|$ 程度です。パウリ行列 $A=\sigma_x$、$B=\sigma_z$ で確かめます。

**反例（$t=1$ だけで偶然成り立つ）**：$A=\begin{pmatrix}0&0\\0&2\pi i\end{pmatrix}$、$B=\begin{pmatrix}0&1\\0&2\pi i\end{pmatrix}$ は $AB\ne BA$ ですが、どれも固有値が $0$ と $2\pi i$ の倍数で対角化できるので、$e^A=e^B=e^{A+B}=I$ です。$e^{2\pi i}=1$ という周期のしわざで、[det のノート](../../../研究ノート/04_群論・代数/det_exp_trace_proof.md)の「1つの $U$ からは $\operatorname{tr}=0$ は出ない」反例と同じ仕組みです。$t$ を少しずらすと成り立たなくなります。

In [ ]:
sx = np.array([[0, 1], [1, 0]], dtype=complex)
sz = np.array([[1, 0], [0, -1]], dtype=complex)
sy = np.array([[0, -1j], [1j, 0]])
comm = sx @ sz - sz @ sx
# パウリ行列：σiσj = δij I + i ε_ijk σk（反交換子 + 交換子）
sig = [sx, sy, sz]
eps3 = np.zeros((3, 3, 3))
for (i, j, k) in ((0, 1, 2), (1, 2, 0), (2, 0, 1)):
    eps3[i, j, k], eps3[j, i, k] = 1, -1
for i in range(3):
    for j in range(3):
        anti = sig[i] @ sig[j] + sig[j] @ sig[i]
        com = sig[i] @ sig[j] - sig[j] @ sig[i]
        assert np.allclose(anti, 2 * (i == j) * np.eye(2))                                   # {σi, σj} = 2δij I
        assert np.allclose(com, 2j * sum(eps3[i, j, k] * sig[k] for k in range(3)))          # [σi, σj] = 2i ε_ijk σk
        assert np.allclose(sig[i] @ sig[j], (i == j) * np.eye(2) + 1j * sum(eps3[i, j, k] * sig[k] for k in range(3)))
assert np.allclose(comm, -2j * sy)
ts = np.logspace(-3, -0.5, 15)
gap = np.array([np.linalg.norm(expm(t * sx) @ expm(t * sz) - expm(t * (sx + sz)), 2) for t in ts])
pred = ts**2 / 2 * np.linalg.norm(comm, 2)
slope = np.polyfit(np.log(ts[:8]), np.log(gap[:8]), 1)[0]
print(f"ずれの両対数の傾き = {slope:.3f}（t² に比例なら 2）、t = 0.001 でのずれ / (t²/2 ‖[A,B]‖) = {gap[0] / pred[0]:.4f}")
assert abs(slope - 2) < 0.05 and abs(gap[0] / pred[0] - 1) < 1e-2
# 交換する場合（B = 2A + I）はぴったり一致
B2 = 2 * sx + np.eye(2)
assert np.allclose(expm(sx) @ expm(B2), expm(sx + B2))
# 反例：交換しないのに、t = 1 だけで e^(A+B) = e^A e^B
Ac = np.array([[0, 0], [0, 2j * np.pi]])
Bc = np.array([[0, 1], [0, 2j * np.pi]])
assert not np.allclose(Ac @ Bc, Bc @ Ac)
assert np.allclose(expm(Ac), np.eye(2)) and np.allclose(expm(Bc), np.eye(2)) and np.allclose(expm(Ac + Bc), np.eye(2))
assert not np.allclose(expm(0.9 * Ac) @ expm(0.9 * Bc), expm(0.9 * (Ac + Bc)))      # t = 0.9 では成り立たない
print("反例：AB ≠ BA なのに e^A = e^B = e^(A+B) = I（t = 0.9 にずらすと不一致）")
fig = go.Figure([go.Scatter(x=ts.tolist(), y=gap.tolist(), mode="lines+markers", name="‖e^(tA)e^(tB) − e^(t(A+B))‖"),
                 go.Scatter(x=ts.tolist(), y=pred.tolist(), mode="lines", line=dict(dash="dash"), name="(t²/2)‖[A,B]‖")])
fig.update_layout(height=360, width=780, xaxis_type="log", yaxis_type="log", xaxis_title="t", margin=dict(t=20))
fig

## 7. トロッター分解：細かく刻んで交互に掛ける

§6 のずれは $t^2$ に比例するので、時間 $t$ を $n$ 等分すると、1回あたりのずれは $(t/n)^2$ 程度、$n$ 回分で $t^2/n$ 程度になります。したがって

$$
e^{(A+B)t}\approx\big(e^{At/n}e^{Bt/n}\big)^n\qquad(\text{誤差}\propto1/n)
$$

で、$n$ を増やせばいくらでも正確になります（**トロッター分解**）。さらに $e^{At/2n}e^{Bt/n}e^{At/2n}$ と対称に並べると、誤差は $1/n^2$ で小さくなります（**ストラング分解**、2次のスズキ・トロッター分解）。

**量子コンピュータでの役割**：ハミルトニアン $H$ が $H=H_1+H_2$ のような和で、それぞれの $e^{-iH_kt}$ は簡単なゲート（[NB-02](../../02_su2_rotation/02_bloch_rotation.ipynb) の回転ゲート）で作れるとき、全体の時間発展 $e^{-iHt}$ をトロッター分解で回路にします。ここでは $H=\sigma_x+\sigma_z$ とし、Qiskit の `PauliEvolutionGate` と `LieTrotter`・`SuzukiTrotter` で作った回路とも比べます。誤差の上限の導出、ストラング分解で2次の誤差が消える理由、$e^{-iHt}$ がユニタリである理由は、[行列の指数関数と交換子のノート](../../../研究ノート/04_群論・代数/matrix_exponential_commutator_bch_trotter.md)の Part VI にあります。

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.quantum_info import Operator, SparsePauliOp
from qiskit.synthesis import LieTrotter, SuzukiTrotter

t_total = 1.0
U_exact = expm(-1j * t_total * (sx + sz))


def lie_trotter(n: int) -> Matrix:
    d = t_total / n
    return np.linalg.matrix_power(expm(-1j * d * sz) @ expm(-1j * d * sx), n)      # 先に σx、次に σz


def strang(n: int) -> Matrix:
    d = t_total / n
    return np.linalg.matrix_power(expm(-1j * d / 2 * sx) @ expm(-1j * d * sz) @ expm(-1j * d / 2 * sx), n)


def qiskit_trotter(synthesis: object) -> Matrix:
    qc = QuantumCircuit(1)
    qc.append(PauliEvolutionGate(SparsePauliOp(["X", "Z"], [1.0, 1.0]), time=t_total, synthesis=synthesis), [0])
    return Operator(qc.decompose().decompose()).data


def same_up_to_phase(U: Matrix, V: Matrix) -> bool:
    return bool(np.isclose(abs(np.trace(U.conj().T @ V)), U.shape[0]))


ns = np.array([1, 2, 4, 8, 16, 32, 64, 128])
err_lie = np.array([np.linalg.norm(lie_trotter(int(n)) - U_exact, 2) for n in ns])
err_strang = np.array([np.linalg.norm(strang(int(n)) - U_exact, 2) for n in ns])
for n in (2, 8, 32):
    assert same_up_to_phase(qiskit_trotter(LieTrotter(reps=n)), lie_trotter(n))
    assert same_up_to_phase(qiskit_trotter(SuzukiTrotter(order=2, reps=n)), strang(n))
s1 = np.polyfit(np.log(ns[3:]), np.log(err_lie[3:]), 1)[0]
s2 = np.polyfit(np.log(ns[3:]), np.log(err_strang[3:]), 1)[0]
print(f"誤差の傾き：トロッター {s1:.2f}（理論 −1）、ストラング {s2:.2f}（理論 −2）")
print("Qiskit の LieTrotter・SuzukiTrotter(order=2) の回路は、自前の計算と（大域位相を除いて）一致")
assert abs(s1 + 1) < 0.1 and abs(s2 + 2) < 0.1
fig = go.Figure([go.Scatter(x=ns.tolist(), y=err_lie.tolist(), mode="lines+markers", name="トロッター（誤差 ∝ 1/n）"),
                 go.Scatter(x=ns.tolist(), y=err_strang.tolist(), mode="lines+markers", name="ストラング（誤差 ∝ 1/n²）")])
fig.update_layout(height=380, width=780, xaxis_type="log", yaxis_type="log", xaxis_title="分割数 n",
                  yaxis_title="‖近似 − e^(−iHt)‖", margin=dict(t=20))
fig

## 観察のポイント（まとめ）

- 行列の指数関数は、$e^x$ の級数に行列を入れたもので、部分和は必ず収束する（行列が大きいほど、多くの項が必要）。
- $J^2=-I$ の $J$ では $e^{\theta J}$ が回転行列になり、オイラーの公式の行列版になる。
- $(PAP^{-1})^k=PA^kP^{-1}$ から $Pe^AP^{-1}=e^{PAP^{-1}}$ が出て、対角化できれば固有値の指数関数だけで計算できる。
- 微分は1次の近似で、捨てる部分は $O(\varepsilon^2)$。行列式の1次の係数が Jacobi の公式で、$\det(e^A)=e^{\operatorname{tr}A}$ の証明の要になる。
- 交換しない行列では $e^{A+B}\ne e^Ae^B$ で、ずれは $\frac{t^2}2[A,B]$ から始まる。細かく刻めば（トロッター分解）、いくらでも正確に近似でき、量子回路で時間発展を作る基本になる。

**次のノートブック**：[03_jacobian_linearization.ipynb](03_jacobian_linearization.ipynb) で、多変数の1次の近似（ヤコビ行列）と、それが壊れる場所（$\det J=0$）を見ます。